# Auditoría Meta 2021–2026 · Mushuc Runa

Notebook complementario del informe ejecutivo. Procesa únicamente los archivos locales ya extraídos; **no llama a Meta** y no contiene credenciales.

Alcance: 1.414 publicaciones públicas de dos páginas y 13 meses parciales de pauta de una cuenta. Las interacciones orgánicas son conteos absolutos, no tasas de engagement. Las acciones publicitarias son atribuidas por Meta y no equivalen a ventas conciliadas.


In [1]:
from pathlib import Path
import csv, json

BASE = Path.cwd()
if not (BASE / "datos").exists():
    candidates = list(Path.cwd().glob("**/2026-08-24_auditoria-meta-2021-2026/datos"))
    if not candidates:
        raise FileNotFoundError("Abre el notebook desde su carpeta o desde la raíz del repositorio")
    BASE = candidates[0].parent
DATA = BASE / "datos"

def read_csv(name):
    with (DATA / name).open(encoding="utf-8") as handle:
        return list(csv.DictReader(handle))

def show(rows, columns, limit=None):
    rows = rows[:limit] if limit else rows
    widths = {c: max(len(c), *(len(str(r.get(c, ""))) for r in rows)) for c in columns}
    print(" | ".join(c.ljust(widths[c]) for c in columns))
    print("-+-".join("-" * widths[c] for c in columns))
    for row in rows:
        print(" | ".join(str(row.get(c, "")).ljust(widths[c]) for c in columns))

organic = read_csv("resumen_eventos_organico.csv")
paid = read_csv("pauta_eventos_parcial.csv")
phases = read_csv("resumen_fases.csv")
formats = read_csv("resumen_formatos.csv")
ranking = read_csv("ranking_publicaciones.csv")
quality = json.loads((DATA / "calidad_datos.json").read_text(encoding="utf-8"))
print(f"Datos cargados desde: {DATA}")


Datos cargados desde: /Users/afnaranjo/Documents/ChatGPT/Complejo Muchuc Runa/vault/13_datos-medicion/2026-08-24_auditoria-meta-2021-2026/datos


## 1. Calidad y cobertura

Primero se comprueban unicidad, cobertura por página y limitaciones de pauta. Esta es la barrera contra conclusiones falsas.


In [2]:
print("Publicaciones:", quality["publicaciones_fuente"])
print("IDs únicos:", quality["ids_unicos"])
print("IDs duplicados:", quality["ids_duplicados"])
print("Mensajes vacíos:", quality["mensajes_vacios"])
print("Publicaciones clasificadas en eventos:", quality["publicaciones_en_eventos_clasificados"])
for page_id, info in quality["cobertura_por_pagina"].items():
    print(f"- {info['pagina'].strip()}: {info['filas']} filas, {info['primera_publicacion_utc']} → {info['ultima_publicacion_utc']}")
print("Pauta:", quality["pauta"]["filas_mensuales"], "meses; cuentas:", quality["pauta"]["cuentas_presentes"])
print("Detención segura:", quality["pauta"]["detencion"])


Publicaciones: 1414
IDs únicos: 1414
IDs duplicados: 0
Mensajes vacíos: 72
Publicaciones clasificadas en eventos: 1363
- Carnavales Mushuc Runa: 1017 filas, 2021-01-23T13:18:56+0000 → 2026-03-13T23:35:53+0000
- Finados Mushuc Runa: 397 filas, 2025-09-06T01:42:24+0000 → 2025-11-19T01:00:53+0000
Pauta: 13 meses; cuentas: ['ExpoFeria Mushuc Runa']
Detención segura: Meta detuvo la consulta: HTTP 403; código 4; Application request limit reached


## 2. Rendimiento orgánico comparable

La mediana es más útil que el total para comparar una publicación típica, porque unos pocos contenidos concentran gran parte de las interacciones.


In [3]:
show(organic, [
    "etiqueta", "publicaciones", "publicaciones_por_dia_activo",
    "interacciones_publicas_total", "interacciones_mediana",
    "porcentaje_en_copias_repetidas"
])


etiqueta      | publicaciones | publicaciones_por_dia_activo | interacciones_publicas_total | interacciones_mediana | porcentaje_en_copias_repetidas
--------------+---------------+------------------------------+------------------------------+-----------------------+-------------------------------
Finados 2021  | 14            | 2.0                          | 486                          | 18.0                  | 35.7                          
Carnaval 2022 | 156           | 4.33                         | 88943                        | 101.5                 | 5.1                           
Finados 2022  | 6             | 1.5                          | 2376                         | 131.5                 | 33.3                          
Carnaval 2023 | 241           | 4.55                         | 96138                        | 99                    | 16.2                          
Carnaval 2024 | 123           | 2.67                         | 89458                        | 203         

## 3. Pauta parcial por evento

Los montos son **gasto mínimo observado** en la cuenta ExpoFeria. No incluyen la cuenta Complejo, campañas desglosadas ni períodos anteriores a agosto de 2023. El alcance sumado entre meses no está deduplicado.


In [4]:
show(paid, [
    "etiqueta", "meses_incluidos", "gasto_observado_usd", "impresiones",
    "clics_enlace", "cpc_enlace_usd", "conversaciones_iniciadas_7d",
    "costo_conversacion_usd", "compras_meta_atribuidas"
])


etiqueta      | meses_incluidos                 | gasto_observado_usd | impresiones | clics_enlace | cpc_enlace_usd | conversaciones_iniciadas_7d | costo_conversacion_usd | compras_meta_atribuidas
--------------+---------------------------------+---------------------+-------------+--------------+----------------+-----------------------------+------------------------+------------------------
Carnaval 2024 | 2023-12|2024-01|2024-02|2024-03 | 5435.75             | 15175162    | 116439       | 0.047          | 309.0                       | 17.59                  | 34.0                   
Carnaval 2025 | 2025-01|2025-02|2025-03         | 9730.14             | 26233034    | 93399        | 0.104          | 2460.0                      | 3.96                   | 25.0                   
Finados 2025  | 2025-10|2025-11                 | 3531.42             | 7443415     | 49861        | 0.071          | 1083.0                      | 3.26                   | 221.0                  
Carnaval 2026 |

## 4. Saturación de Finados 2025

La fase en vivo concentró 179 publicaciones en cinco días. La suma de interacciones creció por volumen y por una pieza atípica, mientras la publicación típica quedó en 33 interacciones.


In [5]:
finados_phases = [row for row in phases if row["evento"] == "finados-2025"]
show(finados_phases, [
    "fase", "publicaciones", "dias_con_publicacion", "publicaciones_por_dia_activo",
    "interacciones_total", "interacciones_mediana"
])


fase          | publicaciones | dias_con_publicacion | publicaciones_por_dia_activo | interacciones_total | interacciones_mediana
--------------+---------------+----------------------+------------------------------+---------------------+----------------------
consideracion | 38            | 18                   | 2.11                         | 9741                | 86.0                 
conversion    | 144           | 21                   | 6.86                         | 44114               | 40.0                 
urgencia      | 56            | 7                    | 8.0                          | 3152                | 29.5                 
en-vivo       | 179           | 5                    | 35.8                         | 51959               | 33                   
postevento    | 13            | 5                    | 2.6                          | 1920                | 73                   


## 5. Formatos

El video tuvo una mediana superior a la foto en cada edición con una muestra suficiente. Esto respalda una estrategia *video-first*, no una mayor frecuencia de publicación.


In [6]:
selected = [row for row in formats if row["evento"] in {
    "carnaval-2022", "carnaval-2023", "carnaval-2024", "carnaval-2025", "finados-2025", "carnaval-2026"
} and row["formato"] in {"added_video", "added_photos"}]
show(selected, ["evento", "formato", "publicaciones", "interacciones_mediana", "compartidos_mediana"])


evento        | formato      | publicaciones | interacciones_mediana | compartidos_mediana
--------------+--------------+---------------+-----------------------+--------------------
carnaval-2022 | added_photos | 93            | 88                    | 12                 
carnaval-2022 | added_video  | 56            | 211.0                 | 27.5               
carnaval-2023 | added_photos | 158           | 91.0                  | 14.0               
carnaval-2023 | added_video  | 72            | 179.0                 | 29.0               
carnaval-2024 | added_photos | 70            | 115.5                 | 18.5               
carnaval-2024 | added_video  | 43            | 450                   | 73                 
carnaval-2025 | added_photos | 73            | 80                    | 9                  
carnaval-2025 | added_video  | 52            | 324.5                 | 36.5               
finados-2025  | added_photos | 194           | 34.5                  | 4.5                

## 6. Piezas líderes de Finados 2025

El ranking identifica señales creativas, no causalidad. No contiene comentarios individuales ni nombres de usuarios.


In [7]:
top = [row for row in ranking if row["evento"] == "finados-2025"][:10]
show(top, ["rango", "fecha_guayaquil", "formato", "interacciones_publicas", "comentarios", "compartidos", "mensaje_resumen"])


rango | fecha_guayaquil | formato      | interacciones_publicas | comentarios | compartidos | mensaje_resumen                                                                                                                                                                                                                                                                                             
------+-----------------+--------------+------------------------+-------------+-------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
1     | 2025-11-04      | added_video  | 28595                  | 113         | 1403        | Dian Paucar se apoderó del megaescenario y nos regaló un cierre inolvidable.                                        

## 7. Pruebas de integridad

Estas aserciones permiten detectar duplicados, cifras negativas o cambios involuntarios en los datos derivados.


In [8]:
assert quality["publicaciones_fuente"] == quality["ids_unicos"]
assert quality["ids_duplicados"] == 0
assert all(float(row["gasto_observado_usd"]) >= 0 for row in paid)
assert all(int(row["publicaciones"]) > 0 for row in organic)
assert {row["evento"] for row in paid} == {"carnaval-2024", "carnaval-2025", "finados-2025", "carnaval-2026"}
print("VALIDACIÓN OK: integridad estructural y controles mínimos superados")


VALIDACIÓN OK: integridad estructural y controles mínimos superados


## Conclusión reproducible

Los datos respaldan cinco decisiones: reducir saturación, producir video con propósito, usar música como puerta de entrada sin convertirla en toda la marca, construir confianza operativa y no optimizar a “compra” hasta reconciliar píxel, boletería y facturación. La estrategia completa se documenta en el informe canónico y en la directriz digital 2026.
